In [0]:
CREATE SCHEMA IF NOT EXISTS training_catalog.silver;
USE training_catalog.silver;

In [0]:
CREATE OR REPLACE TABLE silver_customers
USING DELTA
AS
SELECT
    CAST(CustomerID AS INT) AS CustomerID,
    INITCAP(TRIM(CustomerName)) AS CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    TO_DATE(LastUpdated,'dd-MM-yyyy') AS LastUpdated
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY CustomerID
               ORDER BY TO_DATE(LastUpdated,'dd-MM-yyyy') DESC
           ) rn
    FROM training_catalog.bronze.customers_raw
    WHERE CustomerID IS NOT NULL
      AND CustomerName IS NOT NULL
      AND Email IS NOT NULL
)
WHERE rn=1;

In [0]:
CREATE OR REPLACE TABLE silver_products
USING DELTA
AS
SELECT DISTINCT
    CAST(ProductID AS INT) AS ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    CAST(UnitPrice AS DECIMAL(10,2)) AS UnitPrice
FROM training_catalog.bronze.products_raw
WHERE ProductID IS NOT NULL
  AND ProductName IS NOT NULL
  AND UnitPrice IS NOT NULL
  AND CAST(UnitPrice AS DECIMAL(10,2)) > 0;

In [0]:
CREATE OR REPLACE TABLE silver_stores
USING DELTA
AS
SELECT DISTINCT
    CAST(StoreID AS INT) AS StoreID,
    INITCAP(TRIM(StoreName)) AS StoreName,
    TRIM(Region) AS Region
FROM training_catalog.bronze.stores_raw
WHERE StoreID IS NOT NULL
  AND StoreName IS NOT NULL
  AND Region IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE silver_sales
USING DELTA
AS
SELECT
    CAST(TransactionID AS INT) AS TransactionID,
    CAST(CustomerID AS INT) AS CustomerID,
    CAST(ProductID AS INT) AS ProductID,
    CAST(StoreID AS INT) AS StoreID,
    CAST(Quantity AS INT) AS Quantity,
    TO_DATE(TxnDate,'dd-MM-yyyy') AS TxnDate
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY TransactionID
               ORDER BY TransactionID
           ) rn
    FROM training_catalog.bronze.sales_raw
)
WHERE rn=1
  AND Quantity > 0
  AND CustomerID IS NOT NULL
  AND ProductID IS NOT NULL
  AND StoreID IS NOT NULL;

In [0]:
SELECT 'customers' AS table_name, COUNT(*) FROM silver_customers
UNION ALL
SELECT 'products', COUNT(*) FROM silver_products
UNION ALL
SELECT 'stores', COUNT(*) FROM silver_stores
UNION ALL
SELECT 'sales', COUNT(*) FROM silver_sales;

In [0]:
USE training_catalog.silver;

In [0]:
CREATE OR REPLACE TABLE processed_customers
USING DELTA
LOCATION 's3://sales-analytics-lakehouse/processed/customers/'
AS
SELECT * FROM silver_customers;

In [0]:
CREATE OR REPLACE TABLE processed_products
USING DELTA
LOCATION 's3://sales-analytics-lakehouse/processed/products/'
AS
SELECT * FROM silver_products;

In [0]:
CREATE OR REPLACE TABLE processed_stores
USING DELTA
LOCATION 's3://sales-analytics-lakehouse/processed/stores/'
AS
SELECT * FROM silver_stores;

In [0]:
CREATE OR REPLACE TABLE processed_sales
USING DELTA
LOCATION 's3://sales-analytics-lakehouse/processed/sales/'
AS
SELECT * FROM silver_sales;

In [0]:
SELECT 'customers' AS table_name, COUNT(*) FROM silver_customers
UNION ALL
SELECT 'products', COUNT(*) FROM silver_products
UNION ALL
SELECT 'stores', COUNT(*) FROM silver_stores
UNION ALL
SELECT 'sales', COUNT(*) FROM silver_sales;